# Aula 10 - Avaliação Integrada do Módulo 1: Motor de Intertravamento e Diagnóstico
## SCADA-Core Automática — Grupo 04: Classificação e Seleção de Grãos por Visão Computacional
**Estudante com Foco em Matemática Discreta e Teoria da Computação**

Neste notebook realizamos a consolidação e validação integral do **Módulo 1: Lógica Formal & Sistemas Especialistas**, integrando:
1. **Catálogo e Telemetria de Tags ISA-5.1 com Conversão 4–20 mA e Padrão NAMUR NE43:** Conversão afim linear e supervisão física de integridade de laço (*broken wire* e curto-circuito).
2. **Motor de Intertravamento e Verificação Formal de Tautologias de Segurança:** Implementação da lógica de segurança de processo e prova computacional exaustiva ($2^n$ estados) dos teoremas de segurança.
3. **Base de Conhecimento Especialista e Motor de Inferência Forward Chaining:** Mecanismo dedutivo com resolução determinística de conflitos por severidade e garantia de convergência no Menor Modelo de Herbrand (Teorema do Ponto Fixo de Knaster-Tarski).
4. **Suíte Completa de Testes de Estresse:** Bateria de testes industriais validando 100% dos cenários operacionais da planta de seleção de grãos.


In [1]:
import math
import itertools
from dataclasses import dataclass, field
from enum import Enum
from typing import Dict, List, Tuple, Optional, Set, Any

def formatar_tabela(dados: List[Dict[str, Any]]) -> str:
    """Formata uma lista de dicionários em tabela ASCII pura perfeitamente alinhada."""
    if not dados:
        return "Tabela Vazia"
    colunas = list(dados[0].keys())
    larguras = {c: len(str(c)) for c in colunas}
    for row in dados:
        for c in colunas:
            larguras[c] = max(larguras[c], len(str(row.get(c, ""))))
    header = " | ".join(f"{c:<{larguras[c]}}" for c in colunas)
    divisor = "-+-".join("-" * larguras[c] for c in colunas)
    linhas = [header, divisor]
    for row in dados:
        linhas.append(" | ".join(f"{str(row.get(c, '')):<{larguras[c]}}" for c in colunas))
    return "\n".join(linhas)

print("[OK] Ambiente inicializado e funções utilitárias carregadas!")


[OK] Ambiente inicializado e funções utilitárias carregadas!


## 1. Telemetria Industrial ISA-5.1 e Padrão NAMUR NE43 (Conversão 4–20 mA)

A instrumentação analógica transmite grandezas contínuas em miliamperes ($I \in [4.0, 20.0]\text{ mA}$). A conversão para Unidades de Engenharia ($EU$) segue a equação afim:

$$y = f(I) = y_{\min} + \left( \frac{I - 4.0}{16.0} \right) (y_{\max} - y_{\min})$$

Pelo padrão internacional **NAMUR NE43**, correntes fora do intervalo $[3.8, 20.5]\text{ mA}$ disparam sinalizações específicas, sendo $I < 3.6\text{ mA}$ característico de **rompimento de condutor (*Broken Wire*)** e $I > 21.0\text{ mA}$ característico de **curto-circuito**.


In [2]:
class SignalStatus(Enum):
    NORMAL = 1
    UNDER_RANGE = 2
    OVER_RANGE = 3
    BROKEN_WIRE = 4
    SHORT_CIRCUIT = 5

@dataclass
class AnalogTransmitter:
    tag: str
    description: str
    eu_min: float
    eu_max: float
    unit: str

    def evaluate(self, current_ma: float) -> Tuple[float, SignalStatus, bool]:
        """Converte corrente (mA) em unidade de engenharia e avalia status elétrico NAMUR NE43.
        Retorna: (valor_eu, status_eletrico, loop_fault_bool)
        """
        if current_ma < 3.6:
            status = SignalStatus.BROKEN_WIRE
            loop_fault = True
            val_eu = self.eu_min
        elif current_ma < 3.8:
            status = SignalStatus.UNDER_RANGE
            loop_fault = False
            val_eu = self.eu_min
        elif current_ma <= 20.5:
            status = SignalStatus.NORMAL
            loop_fault = False
            ratio = (current_ma - 4.0) / 16.0
            val_eu = self.eu_min + ratio * (self.eu_max - self.eu_min)
        elif current_ma <= 21.0:
            status = SignalStatus.OVER_RANGE
            loop_fault = False
            val_eu = self.eu_max
        else:
            status = SignalStatus.SHORT_CIRCUIT
            loop_fault = True
            val_eu = self.eu_max

        return val_eu, status, loop_fault

TRANSMISSORES = {
    "LIT-101": AnalogTransmitter("LIT-101", "Nível do Funil de Recepção", 0.0, 100.0, "%"),
    "ST-201": AnalogTransmitter("ST-201", "Velocidade da Esteira Transportadora", 0.0, 3.0, "m/s"),
    "WT-301": AnalogTransmitter("WT-301", "Célula de Carga / Massa na Balança", 0.0, 50.0, "kg"),
    "PT-601": AnalogTransmitter("PT-601", "Pressão da Linha Pneumática", 0.0, 10.0, "bar"),
    "LIT-703": AnalogTransmitter("LIT-703", "Nível do Silo de Rejeito Categoria C", 0.0, 100.0, "%"),
}

def processar_telemetria_analogica(leituras_ma: Dict[str, float]) -> Tuple[Dict[str, float], Dict[str, bool], bool]:
    """Processa leituras analógicas em mA e gera valores EU e proposições lógicas."""
    valores_eu = {}
    proposicoes = {}
    alguma_falha_laco = False

    for tag, tx in TRANSMISSORES.items():
        corrente = leituras_ma.get(tag, 12.0)
        val_eu, status, fault = tx.evaluate(corrente)
        valores_eu[tag] = val_eu
        proposicoes[f"falha_laco_{tag}"] = fault
        if fault:
            alguma_falha_laco = True

    # Predicados de limiar (Discretização booleana formal)
    proposicoes["p_NB101"] = (valores_eu["LIT-101"] <= 15.0)  # Nível baixo funil
    proposicoes["p_NA101"] = (valores_eu["LIT-101"] >= 85.0)  # Nível alto funil
    proposicoes["p_NC101"] = (valores_eu["LIT-101"] >= 95.0)  # Nível crítico funil

    proposicoes["p_MOV201"] = (valores_eu["ST-201"] >= 0.10)  # Esteira em movimento
    proposicoes["p_VB201"] = (0.10 <= valores_eu["ST-201"] < 1.20)  # Velocidade baixa
    proposicoes["p_VA201"] = (valores_eu["ST-201"] > 2.50)   # Velocidade alta

    proposicoes["sobrecarga_massa_wt301"] = (valores_eu["WT-301"] > 45.0)  # Sobrecarga
    proposicoes["p_PAL601"] = (valores_eu["PT-601"] < 6.0)   # Pressão pneumática baixa (< 6 bar)

    proposicoes["p_NA703"] = (valores_eu["LIT-703"] >= 80.0)  # Silo de rejeito alto
    proposicoes["p_NC703"] = (valores_eu["LIT-703"] >= 95.0)  # Silo de rejeito crítico (>= 95%)

    proposicoes["falha_laco_geral"] = alguma_falha_laco
    return valores_eu, proposicoes, alguma_falha_laco

# Teste com valores nominais
eu_demo, prop_demo, fault_demo = processar_telemetria_analogica({
    "LIT-101": 12.0, "ST-201": 14.67, "WT-301": 8.8, "PT-601": 15.2, "LIT-703": 6.4
})
print("[OK] Módulo de Telemetria e Padrão NAMUR NE43 compilado com sucesso!")
print(f"Telemetria EU de Exemplo: LIT-101={eu_demo['LIT-101']:.1f}%, ST-201={eu_demo['ST-201']:.2f} m/s, PT-601={eu_demo['PT-601']:.2f} bar")


[OK] Módulo de Telemetria e Padrão NAMUR NE43 compilado com sucesso!
Telemetria EU de Exemplo: LIT-101=50.0%, ST-201=2.00 m/s, PT-601=7.00 bar


## 2. Motor de Intertravamento e Verificação Formal de Tautologias

A segurança funcional do CLP é regida por axiomas booleanos:
1. $\text{Trip}_{\text{GERAL}} \iff (p_{\text{EMERG}} \lor p_{\text{JI201}} \lor p_{\text{PAL601}} \lor p_{\text{NC703}} \lor \neg p_{\text{KSA401}} \lor \text{FalhaLaço})$
2. $c_{\text{PERM}} \iff \neg \text{Trip}_{\text{GERAL}}$
3. $c_{\text{ALIM}} \iff (c_{\text{PERM}} \land p_{\text{MOV201}} \land \neg p_{\text{NB101}})$
4. $c_{\text{FY603}} \iff (p_C \land p_{\text{POS603}} \land \neg p_{\text{PAL601}})$

A classe `TautologyVerifier` avalia exaustivamente o espaço de estados $2^n$ para certificar que os teoremas de segurança não possuem contraexemplos.


In [ ]:
@dataclass
class InterlockOutputs:
    c_PERM: bool
    c_ALIM: bool
    c_ESTEIRA: bool
    c_FY603: bool
    trip_geral: bool

class SafetyInterlockEngine:
    """Motor de Intertravamento Determinístico de Chão de Fábrica (IEC 61131-3)."""
    @staticmethod
    def evaluate(propositions: Dict[str, bool], digital_inputs: Dict[str, bool], ejection_request: bool = False) -> InterlockOutputs:
        p_EMERG = digital_inputs.get("p_EMERG", False)
        p_JI201 = digital_inputs.get("p_JI201", False)
        p_KSA401 = digital_inputs.get("p_KSA401", True)
        falha_laco = propositions.get("falha_laco_geral", False)

        p_PAL601 = propositions.get("p_PAL601", False)
        p_NC703 = propositions.get("p_NC703", False)
        p_MOV201 = propositions.get("p_MOV201", False)
        p_NB101 = propositions.get("p_NB101", False)

        trip_geral = (p_EMERG or p_JI201 or p_PAL601 or p_NC703 or (not p_KSA401) or falha_laco)
        c_PERM = not trip_geral
        c_ALIM = c_PERM and p_MOV201 and (not p_NB101)
        c_ESTEIRA = c_PERM and (not p_JI201) and digital_inputs.get("c_ESTEIRA", True)
        c_FY603 = ejection_request and (not p_PAL601)

        return InterlockOutputs(c_PERM=c_PERM, c_ALIM=c_ALIM, c_ESTEIRA=c_ESTEIRA, c_FY603=c_FY603, trip_geral=trip_geral)

class TautologyVerifier:
    """Verificador Formal de Tautologias de Segurança em Espaço Discreto 2^n."""
    @staticmethod
    def verify_all_theorems() -> List[Dict[str, Any]]:
        results = []

        # Teorema 1: Trip_Geral -> ¬c_ALIM (2^8 = 256 estados)
        t1_valid = True
        for emerg, ji, pal, nc, ksa, laco, mov, nb in itertools.product([False, True], repeat=8):
            trip = (emerg or ji or pal or nc or (not ksa) or laco)
            perm = not trip
            alim = perm and mov and (not nb)
            if trip and alim:
                t1_valid = False
                break
        results.append({
            "Teorema": "Teorema 1: TripGeral -> ¬c_ALIM",
            "Espaço de Estados": "2^8 = 256 estados",
            "Contradições": 0 if t1_valid else 1,
            "Veredito": "TAUTOLOGIA PROVADA" if t1_valid else "FALHA"
        })

        # Teorema 2: ¬(c_FY603 ∧ p_PAL601) (2^2 = 4 estados)
        t2_valid = True
        for req, pal in itertools.product([False, True], repeat=2):
            fy603 = req and (not pal)
            if fy603 and pal:
                t2_valid = False
                break
        results.append({
            "Teorema": "Teorema 2: ¬(c_FY603 ∧ p_PAL601)",
            "Espaço de Estados": "2^2 = 4 estados",
            "Contradições": 0 if t2_valid else 1,
            "Veredito": "TAUTOLOGIA PROVADA" if t2_valid else "FALHA"
        })

        # Teorema 3: c_ALIM -> (p_MOV201 ∧ ¬p_NB101) (2^3 = 8 estados)
        t3_valid = True
        for perm, mov, nb in itertools.product([False, True], repeat=3):
            alim = perm and mov and (not nb)
            if alim and not (mov and (not nb)):
                t3_valid = False
                break
        results.append({
            "Teorema": "Teorema 3: c_ALIM -> (p_MOV201 ∧ ¬p_NB101)",
            "Espaço de Estados": "2^3 = 8 estados",
            "Contradições": 0 if t3_valid else 1,
            "Veredito": "TAUTOLOGIA PROVADA" if t3_valid else "FALHA"
        })

        # Teorema 4: FalhaLaço -> ¬c_PERM ∧ ¬c_ALIM (2^8 = 256 estados)
        t4_valid = True
        for laco, emerg, ji, pal, nc, ksa, mov, nb in itertools.product([False, True], repeat=8):
            trip = (emerg or ji or pal or nc or (not ksa) or laco)
            perm = not trip
            alim = perm and mov and (not nb)
            if laco and (perm or alim):
                t4_valid = False
                break
        results.append({
            "Teorema": "Teorema 4: FalhaLaço -> ¬c_PERM ∧ ¬c_ALIM",
            "Espaço de Estados": "2^8 = 256 estados",
            "Contradições": 0 if t4_valid else 1,
            "Veredito": "TAUTOLOGIA PROVADA" if t4_valid else "FALHA"
        })

        return results

res_taut = TautologyVerifier.verify_all_theorems()
print("=== VERIFICAÇÃO FORMAL DE TAUTOLOGIAS DE SEGURANÇA ===")
print(formatar_tabela(res_taut))


=== VERIFICAÇÃO FORMAL DE TAUTOLOGIAS DE SEGURANÇA ===
Teorema                                    | Espaço de Estados | Contradições | Veredito          
-------------------------------------------+-------------------+--------------+-------------------
Teorema 1: TripGeral -> ¬c_ALIM            | 2^8 = 256 estados | 0            | TAUTOLOGIA PROVADA
Teorema 2: ¬(c_FY603 ∧ p_PAL601)           | 2^2 = 4 estados   | 0            | TAUTOLOGIA PROVADA
Teorema 3: c_ALIM -> (p_MOV201 ∧ ¬p_NB101) | 2^3 = 8 estados   | 0            | TAUTOLOGIA PROVADA
Teorema 4: FalhaLaço -> ¬c_PERM ∧ ¬c_ALIM  | 2^8 = 256 estados | 0            | TAUTOLOGIA PROVADA


## 3. Base de Conhecimento Especialista e Motor de Inferência Forward Chaining

O motor de inferência opera sobre Cláusulas de Horn definidas:
$$\bigwedge_{i=1}^m \phi_i \implies \psi$$

A resolução de conflitos segue determinísticamente a ordenação:
$$\text{Severidade (Crítica > Alta > Média > Baixa)} \succ \text{Especificidade (|Antecedente|)} \succ \text{ID da Regra}$$

A convergência monotônica para o Ponto Fixo de Knaster-Tarski garante que todas as consequências lógicas são inferidas sem loops infinitos.


In [4]:
class Severidade(Enum):
    CRITICA = 1      # Trip de segurança / Parada mandatária
    ALTA = 2         # Falha grave de subsistema / Ejeção inibida
    MEDIA = 3        # Alerta de qualidade / Obstrução de fluxo
    BAIXA = 4        # Alerta preventivo / Calibração / Abastecimento

@dataclass
class Rule:
    rule_id: str
    antecedent: List[Tuple[str, bool]]
    consequent: Tuple[str, bool]
    description: str
    severity: Severidade = Severidade.MEDIA
    action_prescribed: str = ""

    def evaluate_antecedent(self, working_memory: Dict[str, bool]) -> bool:
        for fact_name, expected_val in self.antecedent:
            if fact_name not in working_memory:
                return False
            if working_memory[fact_name] != expected_val:
                return False
        return True

@dataclass
class AuditStep:
    step_number: int
    rule_fired: str
    inferred_fact: str
    inferred_value: bool
    justification_facts: List[str]
    explanation: str

class ForwardChainingEngine:
    """Motor de Inferência Forward Chaining com Ponto Fixo e Resolução de Conflitos."""
    def __init__(self):
        self.knowledge_base: List[Rule] = []
        self.working_memory: Dict[str, bool] = {}
        self.audit_trail: List[AuditStep] = []

    def add_rule(self, rule: Rule) -> None:
        self.knowledge_base.append(rule)

    def load_facts(self, initial_facts: Dict[str, bool]) -> None:
        self.working_memory = initial_facts.copy()
        self.audit_trail.clear()

    def run(self, max_iterations: int = 50) -> Dict[str, bool]:
        iteration = 0
        fired_rule_ids: Set[str] = set()

        while iteration < max_iterations:
            iteration += 1
            candidate_rules: List[Rule] = []

            for rule in self.knowledge_base:
                if rule.rule_id in fired_rule_ids:
                    continue
                if rule.evaluate_antecedent(self.working_memory):
                    conseq_name, conseq_val = rule.consequent
                    if conseq_name not in self.working_memory or self.working_memory[conseq_name] != conseq_val:
                        candidate_rules.append(rule)

            if not candidate_rules:
                break # Ponto Fixo de Knaster-Tarski alcançado

            # Resolução de conflitos: Severidade > Especificidade > ID
            candidate_rules.sort(key=lambda r: (r.severity.value, -len(r.antecedent), r.rule_id))
            selected_rule = candidate_rules[0]

            conseq_name, conseq_val = selected_rule.consequent
            self.working_memory[conseq_name] = conseq_val
            fired_rule_ids.add(selected_rule.rule_id)

            premises_summary = [f"{k}={v}" for k, v in selected_rule.antecedent]
            step = AuditStep(
                step_number=len(self.audit_trail) + 1,
                rule_fired=selected_rule.rule_id,
                inferred_fact=conseq_name,
                inferred_value=conseq_val,
                justification_facts=premises_summary,
                explanation=f"Regra [{selected_rule.rule_id}] disparada: {selected_rule.description}. Ação: {selected_rule.action_prescribed}"
            )
            self.audit_trail.append(step)

        return self.working_memory

    def get_audit_trail_table(self) -> List[Dict[str, Any]]:
        return [
            {
                "Passo": s.step_number,
                "Regra": s.rule_fired,
                "Fato Inferido": f"{s.inferred_fact}={s.inferred_value}",
                "Premissas": ", ".join(s.justification_facts),
                "Explicação Operacional": s.explanation
            }
            for s in self.audit_trail
        ]

def build_knowledge_base() -> ForwardChainingEngine:
    engine = ForwardChainingEngine()

    engine.add_rule(Rule("R01", [("c_ALIM", True), ("p_MOV201", True), ("p_NB101", False), ("p_VAZAO_NULA", True)],
                         ("causa_obstrucao_funil", True),
                         "Obstrução mecânica na saída do funil de recepção", Severidade.MEDIA,
                         "Desligar alimentador e desobstruir grelha de passagem."))

    engine.add_rule(Rule("R02", [("p_NB101", True), ("c_ALIM", True)],
                         ("alerta_funil_baixo", True),
                         "Funil de recepção com baixo nível de matéria-prima", Severidade.BAIXA,
                         "Solicitar recarga imediata de grãos na recepção."))

    engine.add_rule(Rule("R03", [("c_ESTEIRA", True), ("p_JI201", True), ("p_MOV201", False)],
                         ("causa_travamento_esteira", True),
                         "Travamento mecânico no rolo de tração ou motor da esteira", Severidade.CRITICA,
                         "Bloqueio LOTO imediato, inspeção mecânica de mancais e alívio de carga."))

    engine.add_rule(Rule("R04", [("c_ESTEIRA", True), ("p_JI201", False), ("p_MOV201", False)],
                         ("causa_falha_encoder_st201", True),
                         "Falha de sinal no encoder ST-201 ou correia patinando no tambor", Severidade.ALTA,
                         "Inspecionar acoplamento do encoder incremental e esticador da correia."))

    engine.add_rule(Rule("R05", [("sobrecarga_massa_wt301", True)],
                         ("causa_sobrecarga_pesagem", True),
                         "Sobrecarga excessiva de produto sobre a calha de pesagem", Severidade.MEDIA,
                         "Reduzir vibração do alimentador e checar célula de carga WT-301."))

    engine.add_rule(Rule("R06", [("c_ALIM", False), ("p_MOV201", True), ("p_TARA_WT", True)],
                         ("diagnostico_deriva_zero_wt301", True),
                         "Deriva de zero ou impregnação de pó na calha de pesagem", Severidade.BAIXA,
                         "Executar calibração de zero (tara) e limpeza da esteira."))

    engine.add_rule(Rule("R07", [("p_KSA401", False)],
                         ("causa_falha_camera_visao", True),
                         "Falha de comunicação GigE ou software de visão inoperante", Severidade.ALTA,
                         "Reiniciar serviço de visão, checar link de rede e alimentação 24VDC."))

    engine.add_rule(Rule("R08", [("p_TAXA_REJEICAO_ALTA", True)],
                         ("diagnostico_lote_contaminado", True),
                         "Taxa de refugo C acima de 35% (lote com alta impureza/pragas)", Severidade.MEDIA,
                         "Emitir notificação ao controle de qualidade e segregar lote."))

    engine.add_rule(Rule("R09", [("p_KSA401", True), ("p_XS401", True), ("p_REJEICAO_ANOMALA", True)],
                         ("causa_lente_suja_ou_luz", True),
                         "Lente da câmera obstruída por poeira ou luminária LED queimada", Severidade.ALTA,
                         "Limpar vidro protetor da objetiva e testar módulo de iluminação."))

    engine.add_rule(Rule("R10", [("p_PAL601", True)],
                         ("causa_queda_pressao_ar", True),
                         "Pressão pneumática na linha principal abaixo de 6.0 bar", Severidade.ALTA,
                         "Verificar compressor central, dreno de condensado e vazamentos."))

    engine.add_rule(Rule("R11", [("c_FY603", True), ("p_PAL601", False), ("p_ZSH601", False)],
                         ("causa_falha_solenoide_fy603", True),
                         "Válvula solenoide FY-603 não atuou fisicamente apesar do comando elétrico ativo", Severidade.CRITICA,
                         "Testar bobina 24V da solenoide e trocar válvula rápida."))

    engine.add_rule(Rule("R12", [("p_NC703", True)],
                         ("causa_silo_rejeito_cheio", True),
                         "Silo de descarte atingiu 95% de capacidade com risco iminente de transbordo", Severidade.ALTA,
                         "Substituir caçamba de rejeito e resetar permissivo no SCADA."))

    engine.add_rule(Rule("R13", [("falha_laco_LIT-101", True)],
                         ("causa_falha_laco_lit101", True),
                         "Ruptura de fiação ou curto-circuito no transmissor LIT-101 (NAMUR NE43)", Severidade.CRITICA,
                         "Inspecionar laço 4-20mA, borneira e transmissor ultrassônico."))

    engine.add_rule(Rule("R14", [("falha_laco_PT-601", True)],
                         ("causa_falha_laco_pt601", True),
                         "Ruptura de fiação ou curto-circuito no transmissor PT-601 (NAMUR NE43)", Severidade.CRITICA,
                         "Substituir sensor de pressão piezoelétrico e revisar fiação."))

    # Regras de Propagação de Trip Geral
    engine.add_rule(Rule("R_TRIP_EMERG", [("p_EMERG", True)], ("trip_geral", True),
                         "Trip de emergência por botoeira física de soco", Severidade.CRITICA,
                         "Desarme total de atuadores e travamento de segurança."))

    engine.add_rule(Rule("R_TRIP_PNEUM", [("causa_queda_pressao_ar", True)], ("trip_geral", True),
                         "Trip por perda do suprimento de pressão pneumática", Severidade.CRITICA,
                         "Inibir dosagem e ejeção para evitar contaminação do lote nobre."))

    engine.add_rule(Rule("R_TRIP_SILO", [("causa_silo_rejeito_cheio", True)], ("trip_geral", True),
                         "Trip por sobreenchimento do silo de refugo C", Severidade.CRITICA,
                         "Parar alimentação até substituição da caçamba."))

    engine.add_rule(Rule("R_TRIP_LACO", [("falha_laco_geral", True)], ("trip_geral", True),
                         "Trip por falha elétrica de laço em instrumentação 4-20mA (Fail-Safe)", Severidade.CRITICA,
                         "Bloquear processo até normalização da integridade física dos laços."))

    engine.add_rule(Rule("R_BLOQUEIO_ALIM", [("trip_geral", True)], ("bloqueio_alimentador_c_ALIM", True),
                         "Bloqueio imediato do alimentador vibratório por Trip Geral", Severidade.CRITICA,
                         "Garantir c_ALIM = 0 para interromper fluxo de grãos."))

    return engine

print("[OK] Base de Conhecimento R01 a R14 e regras de Trip compiladas!")


[OK] Base de Conhecimento R01 a R14 e regras de Trip compiladas!


## 4. Arquitetura Integrada do SCADA-Core

A classe `SCADACoreSystem` orquestra o pipeline completo a cada ciclo de varredura (*scan cycle*):
$$\text{Telemetria (mA)} \to \text{NAMUR NE43} \to \text{Intertravamento CLP} \to \text{Forward Chaining} \to \text{IHM / SOE}$$


In [5]:
@dataclass
class SCADAResult:
    cenario_id: str
    descricao: str
    valores_eu: Dict[str, float]
    interlocks: InterlockOutputs
    diagnosticos: List[str]
    audit_trail: List[Dict[str, Any]]

class SCADACoreSystem:
    """Sistema SCADA-Core Integrado: Telemetria -> Intertravamento -> Sistema Especialista."""
    def __init__(self):
        self.interlock_engine = SafetyInterlockEngine()
        self.expert_engine = build_knowledge_base()

    def process_cycle(self, cenario_id: str, descricao: str, telemetria_ma: Dict[str, float],
                      entradas_digitais: Dict[str, bool], ejection_request: bool = False,
                      fatos_adicionais: Optional[Dict[str, bool]] = None) -> SCADAResult:
        # 1. Telemetria e NAMUR NE43
        valores_eu, proposicoes, falha_laco = processar_telemetria_analogica(telemetria_ma)

        # 2. Intertravamento Reativo de Segurança (CLP)
        interlocks = self.interlock_engine.evaluate(proposicoes, entradas_digitais, ejection_request)

        # 3. Consolidação dos Fatos na Memória de Trabalho
        fatos_iniciais = {}
        fatos_iniciais.update(proposicoes)
        fatos_iniciais.update(entradas_digitais)
        fatos_iniciais["c_PERM"] = interlocks.c_PERM
        fatos_iniciais["c_ALIM"] = interlocks.c_ALIM
        fatos_iniciais["c_FY603"] = interlocks.c_FY603
        fatos_iniciais["trip_geral"] = interlocks.trip_geral

        # c_ESTEIRA nas regras do sistema especialista representa a solicitação operacional de acionamento do motor
        if "c_ESTEIRA" not in fatos_iniciais:
            fatos_iniciais["c_ESTEIRA"] = interlocks.c_ESTEIRA

        if fatos_adicionais:
            fatos_iniciais.update(fatos_adicionais)

        # 4. Inferência Dedutiva Forward Chaining
        self.expert_engine.load_facts(fatos_iniciais)
        memoria_saturada = self.expert_engine.run()

        # 5. Extração de diagnósticos deduzidos
        diagnosticos = [
            k for k, v in memoria_saturada.items()
            if (k.startswith("causa_") or k.startswith("diagnostico_")) and v is True
        ]

        return SCADAResult(
            cenario_id=cenario_id,
            descricao=descricao,
            valores_eu=valores_eu,
            interlocks=interlocks,
            diagnosticos=diagnosticos,
            audit_trail=self.expert_engine.get_audit_trail_table()
        )

print("[OK] Sistema SCADA-Core Integrado pronto para execução!")


[OK] Sistema SCADA-Core Integrado pronto para execução!


## 5. Suíte de Testes de Estresse Industrial — Validação de 100% dos Cenários

Executamos os 8 cenários industriais de estresse da planta:
- **C01:** Operação Nominal em Regime Permanente
- **C02:** Queda Crítica de Pressão Pneumática ($PT-601 < 6.0\text{ bar}$)
- **C03:** Travamento Mecânico do Motor com Sobrecarga Térmica ($p_{\text{JI201}}=1, p_{\text{MOV201}}=0$)
- **C04:** Rompimento de Cabo 4–20 mA (Broken Wire) no LIT-101 ($I = 2.0\text{ mA}$)
- **C05:** Transbordo Crítico no Silo de Refugo C ($LIT-703 \ge 95\%$)
- **C06:** Falha Física na Válvula Ejetora FY-603 ($ZSH-601 = 0$ com comando ativo)
- **C07:** Obstrução Mecânica na Grelha do Funil de Recepção ($Q_m = 0$)
- **C08:** Avalanche de Múltiplos Alarmes Simultâneos (Estresse Extremo)


In [6]:
def executar_suite_testes_estresse():
    scada = SCADACoreSystem()
    relatorios: List[Dict[str, Any]] = []

    # C01: Nominal
    res1 = scada.process_cycle("C01", "Operação Nominal", 
                               {"LIT-101": 12.0, "ST-201": 14.67, "WT-301": 8.8, "PT-601": 15.2, "LIT-703": 6.4},
                               {"p_EMERG": False, "p_JI201": False, "p_KSA401": True, "p_XS401": True, "p_ZSH601": True, "c_ESTEIRA": True},
                               ejection_request=False, fatos_adicionais={"p_VAZAO_NULA": False, "p_TARA_WT": False})
    assert res1.interlocks.c_PERM is True and res1.interlocks.c_ALIM is True and not res1.interlocks.trip_geral
    relatorios.append({"Cenário": "C01: Regime Nominal", "c_PERM": res1.interlocks.c_PERM, "c_ALIM": res1.interlocks.c_ALIM,
                       "Trip": res1.interlocks.trip_geral, "Diagnósticos": "Nenhum (Operação Saudável)", "Status": "APROVADO [100%]"})

    # C02: Queda Ar
    res2 = scada.process_cycle("C02", "Queda Pressão Ar", 
                               {"LIT-101": 12.0, "ST-201": 14.67, "WT-301": 8.8, "PT-601": 11.2, "LIT-703": 6.4},
                               {"p_EMERG": False, "p_JI201": False, "p_KSA401": True},
                               ejection_request=True)
    assert not res2.interlocks.c_PERM and not res2.interlocks.c_ALIM and not res2.interlocks.c_FY603 and res2.interlocks.trip_geral
    assert "causa_queda_pressao_ar" in res2.diagnosticos
    relatorios.append({"Cenário": "C02: Queda Pressão Ar", "c_PERM": res2.interlocks.c_PERM, "c_ALIM": res2.interlocks.c_ALIM,
                       "Trip": res2.interlocks.trip_geral, "Diagnósticos": ", ".join(res2.diagnosticos), "Status": "APROVADO [100%]"})

    # C03: Travamento Motor
    res3 = scada.process_cycle("C03", "Travamento Mecânico", 
                               {"LIT-101": 12.0, "ST-201": 4.0, "WT-301": 8.8, "PT-601": 15.2, "LIT-703": 6.4},
                               {"p_EMERG": False, "p_JI201": True, "p_KSA401": True, "c_ESTEIRA": True})
    assert not res3.interlocks.c_PERM and not res3.interlocks.c_ALIM and not res3.interlocks.c_ESTEIRA
    assert "causa_travamento_esteira" in res3.diagnosticos
    relatorios.append({"Cenário": "C03: Travamento Motor", "c_PERM": res3.interlocks.c_PERM, "c_ALIM": res3.interlocks.c_ALIM,
                       "Trip": res3.interlocks.trip_geral, "Diagnósticos": ", ".join(res3.diagnosticos), "Status": "APROVADO [100%]"})

    # C04: Cabo Rompido LIT-101
    res4 = scada.process_cycle("C04", "Cabo Rompido LIT-101", 
                               {"LIT-101": 2.0, "ST-201": 14.67, "WT-301": 8.8, "PT-601": 15.2, "LIT-703": 6.4},
                               {"p_EMERG": False, "p_JI201": False, "p_KSA401": True})
    assert not res4.interlocks.c_PERM and not res4.interlocks.c_ALIM and res4.interlocks.trip_geral
    assert "causa_falha_laco_lit101" in res4.diagnosticos
    relatorios.append({"Cenário": "C04: Cabo Rompido LIT-101", "c_PERM": res4.interlocks.c_PERM, "c_ALIM": res4.interlocks.c_ALIM,
                       "Trip": res4.interlocks.trip_geral, "Diagnósticos": ", ".join(res4.diagnosticos), "Status": "APROVADO [100%]"})

    # C05: Silo Rejeito 96%
    res5 = scada.process_cycle("C05", "Silo Rejeito Cheio", 
                               {"LIT-101": 12.0, "ST-201": 14.67, "WT-301": 8.8, "PT-601": 15.2, "LIT-703": 19.36},
                               {"p_EMERG": False, "p_JI201": False, "p_KSA401": True})
    assert not res5.interlocks.c_PERM and not res5.interlocks.c_ALIM and "causa_silo_rejeito_cheio" in res5.diagnosticos
    relatorios.append({"Cenário": "C05: Silo Rejeito 96%", "c_PERM": res5.interlocks.c_PERM, "c_ALIM": res5.interlocks.c_ALIM,
                       "Trip": res5.interlocks.trip_geral, "Diagnósticos": ", ".join(res5.diagnosticos), "Status": "APROVADO [100%]"})

    # C06: Solenoide FY-603
    res6 = scada.process_cycle("C06", "Falha Solenoide FY-603", 
                               {"LIT-101": 12.0, "ST-201": 14.67, "WT-301": 8.8, "PT-601": 15.2, "LIT-703": 6.4},
                               {"p_EMERG": False, "p_JI201": False, "p_KSA401": True, "p_ZSH601": False},
                               ejection_request=True)
    assert "causa_falha_solenoide_fy603" in res6.diagnosticos
    relatorios.append({"Cenário": "C06: Falha Solenoide FY-603", "c_PERM": res6.interlocks.c_PERM, "c_ALIM": res6.interlocks.c_ALIM,
                       "Trip": res6.interlocks.trip_geral, "Diagnósticos": ", ".join(res6.diagnosticos), "Status": "APROVADO [100%]"})

    # C07: Obstrução Funil
    res7 = scada.process_cycle("C07", "Obstrução Funil", 
                               {"LIT-101": 12.0, "ST-201": 14.67, "WT-301": 4.0, "PT-601": 15.2, "LIT-703": 6.4},
                               {"p_EMERG": False, "p_JI201": False, "p_KSA401": True},
                               fatos_adicionais={"p_VAZAO_NULA": True})
    assert "causa_obstrucao_funil" in res7.diagnosticos
    relatorios.append({"Cenário": "C07: Obstrução Funil", "c_PERM": res7.interlocks.c_PERM, "c_ALIM": res7.interlocks.c_ALIM,
                       "Trip": res7.interlocks.trip_geral, "Diagnósticos": ", ".join(res7.diagnosticos), "Status": "APROVADO [100%]"})

    # C08: Avalanche de Alarmes
    res8 = scada.process_cycle("C08", "Avalanche de Alarmes", 
                               {"LIT-101": 12.0, "ST-201": 4.0, "WT-301": 19.0, "PT-601": 10.0, "LIT-703": 19.68},
                               {"p_EMERG": True, "p_JI201": True, "p_KSA401": False},
                               fatos_adicionais={"p_VAZAO_NULA": True})
    assert not res8.interlocks.c_PERM and not res8.interlocks.c_ALIM and res8.interlocks.trip_geral
    assert len(res8.diagnosticos) >= 3
    relatorios.append({"Cenário": "C08: Estresse Extremo", "c_PERM": res8.interlocks.c_PERM, "c_ALIM": res8.interlocks.c_ALIM,
                       "Trip": res8.interlocks.trip_geral, "Diagnósticos": f"{len(res8.diagnosticos)} causas isoladas", "Status": "APROVADO [100%]"})

    print("=== RELATÓRIO OFICIAL DA SUÍTE DE TESTES DE ESTRESSE (AULA 10) ===")
    print(formatar_tabela(relatorios))
    print("\n[CONQUISTA] 100% DOS TESTES DE INTERTRAVAMENTO E DIAGNÓSTICO FORAM APROVADOS!")
    return res8

res_final = executar_suite_testes_estresse()


=== RELATÓRIO OFICIAL DA SUÍTE DE TESTES DE ESTRESSE (AULA 10) ===
Cenário                     | c_PERM | c_ALIM | Trip  | Diagnósticos                | Status         
----------------------------+--------+--------+-------+-----------------------------+----------------
C01: Regime Nominal         | True   | True   | False | Nenhum (Operação Saudável)  | APROVADO [100%]
C02: Queda Pressão Ar       | False  | False  | True  | causa_queda_pressao_ar      | APROVADO [100%]
C03: Travamento Motor       | False  | False  | True  | causa_travamento_esteira    | APROVADO [100%]
C04: Cabo Rompido LIT-101   | False  | False  | True  | causa_falha_laco_lit101     | APROVADO [100%]
C05: Silo Rejeito 96%       | False  | False  | True  | causa_silo_rejeito_cheio    | APROVADO [100%]
C06: Falha Solenoide FY-603 | True   | True   | False | causa_falha_solenoide_fy603 | APROVADO [100%]
C07: Obstrução Funil        | True   | True   | False | causa_obstrucao_funil       | APROVADO [100%]
C08: Estresse E

## 6. Rastreabilidade Operacional e Relatório de Auditoria (SOE)

Visualização da sequência encadeada de eventos (*Sequence of Events / Audit Trail*) derivada pelo motor de inferência no Cenário de Estresse Extremo C08, evidenciando as explicações determinísticas fornecidas à IHM do SCADA.


In [7]:
print("=== AUDIT TRAIL / SEQUENCE OF EVENTS (CENÁRIO C08 - ESTRESSE EXTREMO) ===")
print(formatar_tabela(res_final.audit_trail))


=== AUDIT TRAIL / SEQUENCE OF EVENTS (CENÁRIO C08 - ESTRESSE EXTREMO) ===
Passo | Regra           | Fato Inferido                    | Premissas                   | Explicação Operacional                                                                                                                                                
------+-----------------+----------------------------------+-----------------------------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------
1     | R_BLOQUEIO_ALIM | bloqueio_alimentador_c_ALIM=True | trip_geral=True             | Regra [R_BLOQUEIO_ALIM] disparada: Bloqueio imediato do alimentador vibratório por Trip Geral. Ação: Garantir c_ALIM = 0 para interromper fluxo de grãos.             
2     | R07             | causa_falha_camera_visao=True    | p_KSA401=False              | Regra [R07] disparada: Falha de comunicação GigE ou software 